# Tutorial 02 — Vectorised sheet resistance from four-point-probe data

This notebook uses a **synthetic** dataset of four-point-probe measurements on five thin films of different thickness. The numbers were generated in the notebook from a chosen resistivity, so the "true" answer is known and the analysis can be checked against it. They are illustrative and are not experimental results.

The physics is a standard thin-film measurement. The programming focus is **vectorisation**: analysing every sample and every current at once with NumPy arrays instead of writing loops.

## Learning objectives

By the end of the notebook, you should be able to:

- hold a whole measurement series in a two-dimensional NumPy array and use its axes deliberately;
- use broadcasting to combine arrays of different shapes without loops;
- fit a straight line to many datasets at once and estimate the slope uncertainty;
- convert a four-point-probe slope into sheet resistance (Ω/sq), resistivity (Ω·cm) and conductivity (S/cm); and
- check a vectorised result against a simple loop and against the physics.

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## 1. The measurement and its formula

In a collinear four-point probe, current $I$ flows through the outer two pins and the voltage $V$ is measured across the inner two. Measuring voltage with separate pins removes most of the contact resistance from the result.

For a film that is thin compared with the pin spacing $s$ and laterally much larger than $s$, the sheet resistance is

$$R_s = \frac{\pi}{\ln 2}\,\frac{V}{I} \approx 4.532\,\frac{V}{I},$$

and the resistivity follows from the film thickness $t$:

$$\rho = R_s\,t, \qquad \sigma = 1/\rho .$$

The thin-film formula is normally used when $t \le s/2$ (Schroder, 2006). Samples that are small or measured near an edge need an extra geometric correction factor, which this tutorial leaves out on purpose.

## 2. Generate the synthetic dataset

Five films are given a single resistivity of $2.0 \times 10^{-4}$ Ω·cm, which is of the right order for a transparent conducting oxide such as indium tin oxide. Each film is "measured" at eight currents, and a small amount of random voltage noise is added. The random generator is seeded so every run gives the same numbers.

Note the shapes: thickness has one value per sample, current has one value per measurement point, and voltage has one value for every combination of the two.

In [ ]:
rng = np.random.default_rng(42)

true_resistivity_ohm_cm = 2.0e-4
spacing_mm = 1.0
thickness_nm = np.array([50.0, 100.0, 150.0, 200.0, 250.0])   # shape (5,)
current_mA = np.linspace(1.0, 8.0, 8)                          # shape (8,)

geometry_factor = np.pi / np.log(2)                            # ≈ 4.532, dimensionless
thickness_cm = thickness_nm * 1e-7                             # 1 nm = 1e-7 cm
true_sheet_ohm_sq = true_resistivity_ohm_cm / thickness_cm     # shape (5,)

# Broadcasting: (5, 1) * (1, 8) -> (5, 8). One row per sample, one column per current.
current_A = current_mA * 1e-3
voltage_V = (true_sheet_ohm_sq[:, None] / geometry_factor) * current_A[None, :]
voltage_V = voltage_V + rng.normal(0.0, 20e-6, size=voltage_V.shape)  # 20 µV noise
voltage_mV = voltage_V * 1e3

print("thickness_nm shape:", thickness_nm.shape)
print("current_mA shape:  ", current_mA.shape)
print("voltage_mV shape:  ", voltage_mV.shape)
print(np.round(voltage_mV, 3))

Before analysing anything, check the assumption behind the formula. The thickest film is 250 nm and the pin spacing is 1 mm, so $t/s$ is tiny. An `assert` makes the notebook stop with an error if someone later changes the data to something the formula does not cover.

In [ ]:
spacing_nm = spacing_mm * 1e6
ratio_t_to_s = thickness_nm / spacing_nm
print(f"Largest t/s: {ratio_t_to_s.max():.1e}")
assert np.all(ratio_t_to_s <= 0.5), "Thin-film formula not valid: t > s/2"


## 3. Fit every sample at once

For each sample we fit $V = aI + b$. The slope $a$ is $V/I$ in ohms. The intercept $b$ should be close to zero; a clear offset in real data can point to thermoelectric voltages or an instrument zero error.

The least-squares slope has a closed form that works along an array axis:

$$a = \frac{\sum (I - \bar I)(V - \bar V)}{\sum (I - \bar I)^2}.$$

Here `axis=1` means "along the currents", so each sample gets its own slope. Its standard uncertainty comes from the scatter of the residuals:

$$u(a) = \sqrt{\frac{\sum r^2 / (n-2)}{\sum (I - \bar I)^2}}.$$

In [ ]:
n_points = current_A.size

i_centred_A = current_A - current_A.mean()                         # shape (8,)
v_centred_V = voltage_V - voltage_V.mean(axis=1, keepdims=True)   # shape (5, 8)

sxx_A2 = np.sum(i_centred_A**2)
slope_ohm = np.sum(i_centred_A * v_centred_V, axis=1) / sxx_A2     # shape (5,)
intercept_V = voltage_V.mean(axis=1) - slope_ohm * current_A.mean()

fitted_V = slope_ohm[:, None] * current_A + intercept_V[:, None]
residual_V = voltage_V - fitted_V
slope_unc_ohm = np.sqrt(np.sum(residual_V**2, axis=1) / (n_points - 2) / sxx_A2)

for t, a, ua, b in zip(thickness_nm, slope_ohm, slope_unc_ohm, intercept_V):
    print(f"t = {t:5.0f} nm   V/I = {a:.4f} ± {ua:.4f} ohm   intercept = {b*1e6:+6.1f} µV")

## 4. Check the vectorised fit against a loop

A fast answer is only useful if it is the right answer. `np.polyfit` fits one sample at a time, so a short loop gives an independent result to compare with.

In [ ]:
loop_slope_ohm = np.array([np.polyfit(current_A, row, deg=1)[0] for row in voltage_V])

print("Largest difference:", np.max(np.abs(loop_slope_ohm - slope_ohm)), "ohm")
assert np.allclose(loop_slope_ohm, slope_ohm), "Vectorised and loop fits disagree"
print("Vectorised and loop fits agree.")

## 5. Convert to sheet resistance, resistivity and conductivity

The film thickness also has an uncertainty. Here it is taken as ±5 %, a reasonable figure for a profilometer or ellipsometer measurement. Because $\rho = R_s t$ is a product, the relative uncertainties add in quadrature.

In [ ]:
thickness_rel_unc = 0.05

sheet_resistance_ohm_sq = geometry_factor * slope_ohm
sheet_unc_ohm_sq = geometry_factor * slope_unc_ohm

resistivity_ohm_cm = sheet_resistance_ohm_sq * thickness_cm
resistivity_rel_unc = np.sqrt((sheet_unc_ohm_sq / sheet_resistance_ohm_sq)**2 + thickness_rel_unc**2)
resistivity_unc_ohm_cm = resistivity_ohm_cm * resistivity_rel_unc

conductivity_S_cm = 1.0 / resistivity_ohm_cm

print(" t (nm)   R_s (ohm/sq)        rho (1e-4 ohm cm)    sigma (S/cm)")
for t, rs, urs, r, ur, c in zip(thickness_nm, sheet_resistance_ohm_sq, sheet_unc_ohm_sq,
                                resistivity_ohm_cm, resistivity_unc_ohm_cm, conductivity_S_cm):
    print(f"{t:6.0f}   {rs:7.3f} ± {urs:5.3f}     {r*1e4:5.3f} ± {ur*1e4:5.3f}       {c:7.0f}")

## 6. Validate against the physics

Two checks are worth making.

1. Because this data is synthetic, the recovered resistivity can be compared with the value it was built from. Every sample should agree within about two standard uncertainties.
2. For a film whose resistivity does not depend on thickness, $R_s = \rho / t$. A plot of $R_s$ against $1/t$ should therefore be a straight line through the origin with slope $\rho$.

In [ ]:
z_score = (resistivity_ohm_cm - true_resistivity_ohm_cm) / resistivity_unc_ohm_cm
print(f"Largest deviation from the true value: {np.max(np.abs(z_score)):.2f} standard uncertainties")
assert np.all(np.abs(z_score) < 2), "A sample disagrees with the known resistivity"

inv_thickness_per_cm = 1.0 / thickness_cm
rho_from_slope_ohm_cm = np.sum(inv_thickness_per_cm * sheet_resistance_ohm_sq) / np.sum(inv_thickness_per_cm**2)
print(f"Resistivity from R_s vs 1/t (line through origin): {rho_from_slope_ohm_cm:.3e} ohm cm")
print(f"Value used to build the data:                       {true_resistivity_ohm_cm:.3e} ohm cm")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))

for t, row_mV, fit_V in zip(thickness_nm, voltage_mV, fitted_V):
    line, = ax1.plot(current_mA, fit_V * 1e3)
    ax1.scatter(current_mA, row_mV, color=line.get_color(), s=18, zorder=3, label=f"{t:.0f} nm")
ax1.set_xlabel("Current (mA)")
ax1.set_ylabel("Inner-probe voltage (mV)")
ax1.set_title("Synthetic four-point-probe data")
ax1.grid(alpha=0.25)
ax1.legend(title="Thickness", fontsize=8)

x_um_inv = inv_thickness_per_cm * 1e-4    # 1/cm -> 1/µm
ax2.errorbar(x_um_inv, sheet_resistance_ohm_sq, yerr=sheet_unc_ohm_sq, fmt="o", label="From fits", zorder=3)
x_line = np.linspace(0, x_um_inv.max() * 1.05, 50)
ax2.plot(x_line, rho_from_slope_ohm_cm * x_line * 1e4, label="Line through origin")
ax2.set_xlabel("1 / thickness (µm$^{-1}$)")
ax2.set_ylabel("Sheet resistance (Ω/sq)")
ax2.set_title("Constant-resistivity check")
ax2.set_xlim(left=0)
ax2.set_ylim(bottom=0)
ax2.grid(alpha=0.25)
ax2.legend()

fig.tight_layout()
plt.show()

## Interpretation and checks

- The vectorised fit gives the same slopes as a loop over `np.polyfit`, so the array code does what it claims.
- All five resistivities agree with the value used to build the data, within their uncertainties. That confirms the unit conversions (nm to cm, mA to A) and the geometry factor are applied correctly.
- The ±5 % thickness uncertainty, not the voltage noise, dominates the resistivity uncertainty here. In practice that tells you where better measurements would help most.
- The straight line through the origin in the right-hand plot is expected only because the synthetic films were given one resistivity. Real thin films often become more resistive as they get thinner, because of grain-boundary and surface scattering, so a curved plot from real data would be a physical result rather than a mistake.
- None of this checks the geometric assumptions. A real measurement would also need the sample size, the probe position relative to the edges, and confirmation that the film is uniform.

## Exercise

1. Increase the voltage noise step by step from 20 µV. Roughly how large must it be before the slope uncertainty, rather than the 5 % thickness uncertainty, dominates the resistivity uncertainty?
2. Add a constant 50 µV offset to every voltage. Does the slope change? Does the intercept? Explain why a four-point measurement uses the slope rather than a single V/I ratio.
3. Make the resistivity depend on thickness, for example $\rho(t) = \rho_0 (1 + 30\,\text{nm}/t)$, and repeat the analysis. How does the $R_s$ against $1/t$ plot change?
4. A bulk sample with $t \gg s$ uses $\rho = 2\pi s\,V/I$ instead. Write that calculation for one sample, and explain why the thin-film formula would be wrong for it.

## References

- Valdes, L. B. (1954). Resistivity measurements on germanium for transistors. *Proceedings of the IRE*, 42(2), 420–427.
- Smits, F. M. (1958). Measurement of sheet resistivities with the four-point probe. *Bell System Technical Journal*, 37(3), 711–718.
- Schroder, D. K. (2006). *Semiconductor Material and Device Characterization* (3rd ed.), chapter 1. Wiley.